# Topic Hierarchy — SynBio Papers (Low → Mid → High)

Builds a three-level hierarchy for the existing **SynBio Papers** topic model
by cutting BERTopic's agglomerative merge tree at two levels:

- **mid** — auto-selected by silhouette over `[HIGH_K_MAX + 1, n_low // 3]`
- **high** — auto-selected by silhouette over `[HIGH_K_MIN, HIGH_K_MAX]`

**Inputs:** `papers_topic_model`, `papers_doc_topics.txt`, `papers_topic_names.txt`, `papers_corpus.txt`, `synbio_openalex.txt`

**Outputs (in `assets/reports/`):**
- `papers_topic_hierarchy_map.tsv` — document-level mapping (`ID, low, mid, high`)
- `papers_topic_name_hierarchy.tsv` — low-level names mapped to `low, mid, high`
- `papers_topic_hierarchy_summary.tsv` — mid/high group summary stats

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd

from aux.paths import REPORTS_DIR, HIGH_K_MIN, HIGH_K_MAX, set_seed
from aux.hierarchy import load_hierarchy_inputs, select_hierarchy_levels, write_hierarchy_reports

set_seed()

# ── CONFIG: SynBio Papers ──────────────────────────────────────────────────
PREFIX = "papers"
ID_COL = "ID"
YEAR_COL = "publication_year"
RAW_FILE = "synbio_openalex.txt"
RENAME_ID_FROM = "id"   # source id column to rename to ID_COL (None if already named)

model, doc_topics, topic_names, corpus, raw = load_hierarchy_inputs(
    PREFIX, id_col=ID_COL, year_col=YEAR_COL, raw_filename=RAW_FILE, rename_id_from=RENAME_ID_FROM,
)
print(f"{PREFIX}: {len(doc_topics):,} docs, {len(topic_names):,} topics, "
      f"outliers={(doc_topics['low'] == -1).sum():,}")

papers: 19,017 docs, 225 topics, outliers=6,221


## 1. Build hierarchy and auto-select mid / high levels

In [3]:
corpus_texts = corpus["text"].astype(str).tolist()
hierarchy_map, sel = select_hierarchy_levels(model, corpus_texts, HIGH_K_MIN, HIGH_K_MAX)

print(f"\nHigh K = {sel['high_k']} (silhouette {sel['high_score']:.4f})  |  "
      f"Mid K = {sel['mid_k']} (silhouette {sel['mid_score']:.4f})")
print("\n--- High-level candidates ---")
display(pd.DataFrame(sel["high_scores"], columns=["high_k", "silhouette"]).sort_values("high_k"))
print("--- Mid-level candidates ---")
pd.DataFrame(sel["mid_scores"], columns=["mid_k", "silhouette"]).sort_values("mid_k")

  Building BERTopic hierarchical merge tree …


100%|██████████| 224/224 [00:07<00:00, 29.37it/s]


  Non-outlier low topics: 225
  Tracking cluster maps for k = 1 … 225
  Captured 224 distinct k-level snapshots
  Scoring high-level k candidates in [4, 11] …
    k=  4  silhouette=0.1179
    k=  5  silhouette=0.1162
    k=  6  silhouette=0.1041
    k=  7  silhouette=0.1187
    k=  8  silhouette=0.1094
    k=  9  silhouette=0.0959
    k= 10  silhouette=0.1000
    k= 11  silhouette=0.1099
  ✓ Selected high-level k = 7 (silhouette = 0.1187)
  Scoring mid-level k candidates in [12, 75] …
    k= 12  silhouette=0.1109
    k= 13  silhouette=0.1129
    k= 14  silhouette=0.1219
    k= 15  silhouette=0.0955
    k= 16  silhouette=0.0891
    k= 17  silhouette=0.0787
    k= 18  silhouette=0.0883
    k= 19  silhouette=0.0883
    k= 20  silhouette=0.0933
    k= 21  silhouette=0.1015
    k= 22  silhouette=0.1093
    k= 23  silhouette=0.1159
    k= 24  silhouette=0.1071
    k= 25  silhouette=0.1065
    k= 26  silhouette=0.1083
    k= 27  silhouette=0.1033
    k= 28  silhouette=0.0868
    k= 29  silhou

,high_k,silhouette
0,4,0.117939
1,5,0.116171
2,6,0.104131
3,7,0.118715
4,8,0.109383
5,9,0.095931
6,10,0.099969
7,11,0.109869


--- Mid-level candidates ---


,mid_k,silhouette
0,12,0.110905
1,13,0.112934
2,14,0.121890
3,15,0.095468
4,16,0.089125
...,...,...
59,71,0.102478
60,72,0.094756
61,73,0.093654
62,74,0.094353


## 2. Build and save the report tables

In [4]:
doc_map, name_map, summary = write_hierarchy_reports(
    doc_topics, topic_names, raw, hierarchy_map,
    id_col=ID_COL, year_col=YEAR_COL, prefix=PREFIX,
)
print(f"Saved 3 hierarchy reports for {PREFIX} → {REPORTS_DIR}")
name_map.head()

  Document map: 19,017 rows (6,221 outliers)
  Name map: 225 topics
  Summary: 21 rows (mid + high)
Saved 3 hierarchy reports for papers → /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports


,global_name,low,mid,high
0,Plant Genetic Engineering Tools,0,0,0
1,Ethics and Society in Synthetic Biology,1,1,1
2,Plant Biosynthesis Pathway Engineering,2,0,0
3,Synthetic Biology for Diagnostics and Therapeu...,3,2,2
4,CRISPR-Cas Systems in Synthetic Biology,4,2,2


## 3. Validation

In [5]:
assert list(doc_map.columns) == [ID_COL, "low", "mid", "high"]
assert list(name_map.columns) == ["global_name", "low", "mid", "high"]
assert {"level", "group_id", "total_count", "avg_publication_year", "median_publication_year"}.issubset(summary.columns)
assert len(doc_map) == len(doc_topics)
assert (doc_map.loc[doc_map["low"] == -1, ["mid", "high"]] == -1).all().all()
assert HIGH_K_MIN <= sel["high_k"] <= HIGH_K_MAX
assert sel["mid_min"] <= sel["mid_k"] <= sel["mid_max"]
print("All validation checks passed ✓")

All validation checks passed ✓


## 4. Hierarchy quality summary

In [6]:
outlier_pct = (doc_topics["low"] == -1).mean() * 100
print(f"Selected high-level K : {sel['high_k']}  (silhouette = {sel['high_score']:.4f})")
print(f"Selected mid-level K  : {sel['mid_k']}  (silhouette = {sel['mid_score']:.4f})")
print(f"Outlier documents     : {outlier_pct:.2f}%")
print(f"Mid-level search range: [{sel['mid_min']}, {sel['mid_max']}]")

Selected high-level K : 7  (silhouette = 0.1187)
Selected mid-level K  : 14  (silhouette = 0.1219)
Outlier documents     : 32.71%
Mid-level search range: [12, 75]
